# Experimentation with model

### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

from transformers import AutoTokenizer, set_seed
import os, torch
import sys, argparse, time
import inspect
import numpy as np
from vllm import LLM, SamplingParams

sys.path.append("src") 
from llama_activate import (
    get_llm,
    run_simulation,
    main, 
    call_visualizations,
    pca_visualize
    
)

from utils.path_manager import PathManager

import utils.metrics as metrics
from utils.path_manager import PathManager
from classes.network import RandomNetwork, SocialDistanceAttachment
import utils.load_personas as lp
import utils.visualization as vis
import utils.reading_in as ri
import warnings

# Suppress annoying warning
warnings.filterwarnings("ignore", category=FutureWarning, module="pandas.io.spss")

# --- Model Configuration ---
llama_model = "meta-llama/Meta-Llama-3.1-8B-Instruct" #bigger model
MODEL_ID = os.environ.get("LLAMA_ID", llama_model)
CACHE_DIR = os.environ.get("TRANSFORMERS_CACHE", None)
SEED = 1234
set_seed(SEED)



In [ ]:
# --- Tokenizer Setup ---
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,  
    cache_dir=CACHE_DIR, 
    use_fast=True, 
)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

os.environ["VLLM_BATCH_INVARIANT"] = "1"

def get_llm():
    """Set up the vLLM engine."""
    print(f"Loading vLLM model: {MODEL_ID}...")
    llm = LLM(
        model=MODEL_ID,
        dtype="bfloat16",
        trust_remote_code=True,
        gpu_memory_utilization=0.90,
        seed=SEED,
    )
    return llm

# Initialize the model
pipe = get_llm()

### Set variables

In [ ]:
# --- Simulation Variables ---
rounds = 200
num_agents = 100
net = "sda"
seeds = [73]
alpha = 1.0
m = 2
degree = 6
p = 0.3
k = 0
dim = 2
save = False

# Define the states you want to run (["basis", "depressed"])
states = ["basis"] 

# Create the Namespace (mimicking argparse)
args = argparse.Namespace(
    net=net,
    rounds=rounds,
    num_agents=num_agents,
    seed=seeds[0], # Initial default, will be overwritten in loop
    seeds=seeds,   # List of seeds
    m=m,
    p=p,
    k=k,
    depressed=False,
    enforce_ngrams=False,
    alpha=alpha,
    degree=degree,
    dim=dim,
    save=save,
    use_saved_network=0, # Set to integer if updating existing
    directed=True
)

## Get Network Features

In [ ]:
all_networks_results = main(args, None, states = ["basis"])
network_results = all_networks_results["basis"][0]
network = network_results["network"]
running_fracs = network_results["running_fracs"]
fracs_dist_step = network_results["fracs_dist_step"]
path_manager = PathManager(network=network)

cd_results = metrics.all_agent_phq9_cd(network, window_size=6, shift=6)

# Get paths
data_path = path_manager.get_run_directory(is_plot=False)
plot_path = path_manager.get_run_directory(is_plot=True)
data_filename = path_manager.get_network_filename()
plot_filename = path_manager.get_plot_name()

vis.plot_agent_cd_heatmaps(network, window=6, cd_results=cd_results, metric_name="PHQ-9", path=plot_path, filename=plot_filename, shift=6)

#Maybe bit ugly here but aggregate degree weighted phq9 scores
deg_w_phq9 = metrics.degree_weighted_mean(network)
vis.plot_degree_weighted_phq9(np.array([deg_w_phq9]), plot_path, plot_filename, save=args.save)


call_visualizations(
    network, 
    plot_path, 
    plot_filename, 
    args, 
    running_fracs, 
    fracs_dist_step)

pca_visualize(all_networks_results, plot_path, plot_filename, args)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def plot_phase_dw_phq9_homophily(network, ax=None, label=None, color=None, path="", filename="", save=False, show_fig=False):
    """
    Phase plot: x = degree-weighted PHQ-9, y = PHQ-9 assortativity (homophily), over time.

    Args:
        network: Network after a run (has all_phq9_sumscores, connections).
        ax: matplotlib axis; if None, create new figure.
        label: Optional legend label.
        color: Optional line color (e.g. from a colormap for comparing runs by alpha).
        path, filename: Used if save=True.
        save, show_fig: Whether to save or show the figure.
    """
    # X-axis: degree-weighted PHQ-9 (one value per round), same as in metrics
    dw_phq9 = metrics.degree_weighted_mean(network)
    min_rounds = len(dw_phq9)

    if network.directed:
        graph = nx.DiGraph()
    else:
        graph = nx.Graph()

    for agent in network.all_agents:
        graph.add_node(agent.ID, mood=0)
    for connection in network.connections:
        graph.add_edge(connection[0].ID, connection[1].ID)

    assortativities = []
    for t in range(min_rounds):
        for agent in network.all_agents:
            graph.nodes[agent.ID]["mood"] = agent.all_phq9_sumscores[t]
        try:
            assortativity = nx.numeric_assortativity_coefficient(graph, "mood")
        except Exception:
            assortativity = np.nan
        assortativities.append(assortativity)

    assortativities = np.array(assortativities)

    if ax is None:
        fig, ax = plt.subplots()
    plot_kw = {"label": label, "alpha": 0.8}
    if color is not None:
        plot_kw["color"] = color
    ax.plot(dw_phq9, assortativities, **plot_kw)
    ax.set_xlabel("Degree-weighted PHQ-9")
    ax.set_ylabel("PHQ-9 assortativity (homophily)")
    ax.set_title("Phase plot: degree-weighted PHQ-9 vs homophily")
    if label is not None:
        ax.legend()
    ax.grid(True, alpha=0.3)
    if save and path and filename:
        ax.figure.savefig(f"{path}/phase_dw_phq9_homophily_{filename}.png", dpi=300, bbox_inches="tight")
    if show_fig:
        plt.show()
    return ax


In [ ]:
# Compare phase plots across different homophily settings (dim, alpha); same degree etc.
# Lines are coloured by alpha value.
import copy


# Define (dim, alpha) combinations to compare (degree and other args stay from `args`)
dims_to_compare = [2]           # e.g. [2], [2, 3]
alphas_to_compare = [1.0]  # e.g. [0.5, 1.0, 2.0]

show_fig = True

def phase_plot_homophily_phq9(dims_to_compare, alphas_to_compare, path="", filename="", save=False, show_fig=False):
    """
    Phase plot: x = degree-weighted PHQ-9, y = PHQ-9 assortativity (homophily), over time.

    Args:
        network: Network after a run (has all_phq9_sumscores, connections).
     """

    fig, ax = plt.subplots(figsize=(8, 6))

    for dim in dims_to_compare:
        for alpha in alphas_to_compare:
            args_run = copy.copy(args)
            args_run.dim = dim
            args_run.alpha = alpha

            # retrieve network (needs to be already run)
            all_networks_results = main(args_run, None, states = ["basis"])
            network_results = all_networks_results["basis"][0]
            network = network_results["network"]

            # plot
            plot_phase_dw_phq9_homophily(
                network, ax=ax,
                label=f"α={alpha}, dim={dim}",
                show_fig=False,
            )
            
            # set axes labels
            ax.set_xlabel("Degree-weighted PHQ-9")
            ax.set_ylabel("PHQ-9 assortativity (homophily)")
            ax.set_title("Phase plot: degree-weighted PHQ-9 vs homophily")
            ax.legend(loc="best", fontsize=8)
            plt.tight_layout()
    if save:
        plt.savefig(f"{path}/phase_plot_homophily_phq9_{filename}.png", dpi=300, bbox_inches="tight")
    if show_fig:
        plt.show()
phase_plot_homophily_phq9(dims_to_compare, alphas_to_compare, path="", filename="", save=False, show_fig=False)

In [ ]:
# create network based on args
network, _, _ = run_simulation(args)
vis.print_network_phq9(network, show_fig=True)

In [ ]:
vis.print_network_phq9(network, show_fig=True)


### Generate network and run simulation

In [ ]:
print("Starting simulation via imported function...")
all_networks_results = sim_script.main(args, pipe, states)

#### Visualize CDS

In [ ]:
# Visualizing the first seed of the first state
target_state = states_to_run[0]
run_index = 0 # 0 for the first seed

if target_state in all_networks_results:
    network_data = all_networks_results[target_state][run_index]
    
    network = network_data["network"]
    running_fracs = network_data["running_fracs"]
    fracs_dist_step = network_data["fracs_dist_step"]
    
    # Initialize PathManager with the specific network instance
    path_manager = PathManager(network=network)
    
    data_path = path_manager.get_run_directory(is_plot=False)
    plot_path = path_manager.get_run_directory(is_plot=True)
    data_filename = path_manager.get_network_filename()
    plot_filename = path_manager.get_plot_name()
    
    print(f"Visualizing results for state: {target_state}")
    
    # Print tweet histories
    metrics.print_histories(network, file_dir=data_path, file_name=data_filename, save=args.save)
    
    # Standard visualizations (CDS, frequency, network,)
    call_visualizations(network, plot_path, plot_filename, args, running_fracs, fracs_dist_step)
    
    # PCA Visualization (if large enough simulation is run)
    # pca_visualize(all_networks_results, plot_path, plot_filename, args)
else:
    print(f"No results found for state {target_state}")